In [5]:
print("all ok")

all ok


In [6]:
import os

for key in ("OPENAI_API_KEY", "OPENROUTER_API_KEY","HUGGINGFACEHUB_API_TOKEN","GOOGLE_API_KEY"):
    value = os.getenv(key)
    if value:
        os.environ[key] = value
    print(f"{key} configured: {bool(value)}")

OPENAI_API_KEY configured: True
OPENROUTER_API_KEY configured: True
HUGGINGFACEHUB_API_TOKEN configured: True
GOOGLE_API_KEY configured: True


In [7]:
# 👇 UNIVERSAL LLM CALLER
#    One function that works with any of the 6 providers.
#    Fill in the ___ parts using what you learned in Section 1.
from openai import OpenAI
def call_llm(provider: str, prompt: str, api_key: str = "", model: str = "") -> str:
    """
    Call any LLM provider with the same interface.

    Args:
        provider : One of: "ollama" | "lmstudio" | "openai" | "anthropic" | "gemini" | "openrouter"
        prompt   : The question or instruction to send to the model
        api_key  : Your API key (leave empty for local providers Ollama and LM Studio)
        model    : Model name — if empty, a sensible default is used for each provider

    Returns:
        The model's response as a plain Python string
    """
    
    # ------------------------------------------------------------------ #
    #  OPENAI — cloud API at api.openai.com                              #
    # ------------------------------------------------------------------ #
    
    
    if provider == "openai":
        OPEN_API_KEY = os.getenv("OPENAI_API_KEY")
        if not OPEN_API_KEY:
            raise ValueError("OPENAI_API_KEY environment variable not set")
        print("Calling OpenAI...")
        client = OpenAI(api_key=OPEN_API_KEY)   # TODO: pass the api_key parameter to the client
        model = model or "gpt-4o-mini"  # cheapest GPT-4 class model
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500
        )
        return resp.choices[0].message.content
    
    # ------------------------------------------------------------------ #
    #  OPENROUTER — cloud gateway to 200+ models                          #
    # ------------------------------------------------------------------ #
    
    if provider == "openrouter":
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if not OPENROUTER_API_KEY:
            raise ValueError("OPENROUTER_API_KEY environment variable not set")
        print("Calling OpenRouter...")
        client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=OPENROUTER_API_KEY,
            default_headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        )
        model = model or "meta-llama/llama-3.3-70b-instruct:free"
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
        )
        return resp.choices[0].message.content
    
    # ------------------------------------------------------------------ #
    #  GEMINI — Google's cloud API, its own SDK                           #
    # ------------------------------------------------------------------ #
    
    elif provider == "gemini":
        GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
        if not GOOGLE_API_KEY:
            raise ValueError("GOOGLE_API_KEY environment variable not set")
        print("Calling Google Gemini...")
        from google import genai
        client_gemini = genai.Client(api_key=GOOGLE_API_KEY)
        model_name = model or "gemini-2.0-flash"
        response = client_gemini.models.generate_content(
            model=model_name,
            contents=[{"type": "input_text", "input_text": {"text": prompt}}],
            )
        # return the text field (fallback to str(response) if attribute missing)
        return getattr(response, "text", None) or str(response)
    
    else:
        raise ValueError(
            f"Unknown provider: '{provider}'. "
            f"Choose from: ollama, lmstudio, openai, anthropic, gemini, openrouter"
        )


# 👇 Quick smoke test — run this to verify Ollama works
#    (swap "ollama" for "gemini" or "openrouter" if you're on Colab)
#result = call_llm("openrouter", "What is 2 + 2? Answer with one word only.")
#print(f"Test passed! openai says: {result}")
        
        
        


In [10]:
import time

# 👇 The SAME question will be sent to every provider you configure below
QUESTION = "What is the most important thing to understand about large language models?"
MY_PROVIDERS = {
    #"openai": "gpt-4o-mini",
    "openrouter": "meta-llama/llama-3.3-70b-instruct:free",
    #"gemini": "gemini-2.0-flash"
}

print(f"Question: {QUESTION}")
print("=" * 60)

for provider, model in MY_PROVIDERS.items():
    print(f"\n=== Testing {provider} with model {model} ===")
    start_time = time.time()
    try:
        answer = call_llm(provider, QUESTION, model=model)
        elapsed = time.time() - start_time
        print(f"Answer from {provider} (took {elapsed:.2f} seconds):\n{answer}")
    except Exception as e:
        print(f"Error calling {provider}: {e}")

Question: What is the most important thing to understand about large language models?

=== Testing openrouter with model meta-llama/llama-3.3-70b-instruct:free ===
Calling OpenRouter...
Error calling openrouter: Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}
